# 🔺 Delta Lake — Time Travel & Vacuum

---

## 1. O que é Time Travel?

**Time Travel** é a capacidade do Delta Lake de consultar ou restaurar versões anteriores de uma tabela.

Cada operação (`INSERT`, `UPDATE`, `DELETE`, `MERGE`…) gera um novo commit no `_delta_log`.  
Cada commit é uma **versão numerada** da tabela. O Time Travel permite navegar por essas versões como se fossem snapshots no tempo.

```
  versão 0  →  CREATE TABLE
  versão 1  →  INSERT (1, 'Back')
  versão 2  →  INSERT (2, 'To the')
  versão 3  →  INSERT (3, 'Future')   ◄── estado desejado
  versão 4  →  DELETE FROM demo       ◄── estado atual (tabela vazia)
                    │
                    ▼
         SELECT * FROM demo@v3  →  retorna dados da versão 3
         RESTORE TABLE demo TO VERSION AS OF 3  →  volta para v3
```

---

### Por que isso é possível?

Lembre-se: o Delta Lake **não apaga arquivos Parquet imediatamente**.  
Um `DELETE` apenas marca linhas com Deletion Vectors ou adiciona uma entrada `remove` no `_delta_log`.  
Os arquivos físicos continuam no disco — e é exatamente isso que permite "voltar no tempo".

```
  Disco:
  ├── part-0001.snappy.parquet  (contém: 'Back')      ← ainda existe
  ├── part-0002.snappy.parquet  (contém: 'To the')    ← ainda existe
  ├── part-0003.snappy.parquet  (contém: 'Future')    ← ainda existe
  └── _delta_log/
        ├── 00000.json  CREATE
        ├── 00001.json  INSERT → add part-0001
        ├── 00002.json  INSERT → add part-0002
        ├── 00003.json  INSERT → add part-0003
        └── 00004.json  DELETE → remove part-0001, 0002, 0003

  Query @v3: ignora o json de versão 4, lê os três parquets → dados completos ✓
```

---

### O problema que isso cria

Como os arquivos nunca são removidos automaticamente, com o tempo acumulam-se:
- Arquivos `.parquet` de versões antigas (órfãos após `remove`)
- Arquivos `.json` do `_delta_log` de cada commit

Em tabelas com alto volume de operações, isso pode atingir o limite de armazenamento e impedir a criação de novos arquivos.  
É aqui que entra o **`VACUUM`** — visto na seção 4.

---

### Mecanismos de controle de retenção

O Delta Lake expõe duas propriedades para controlar por quanto tempo os dados históricos são mantidos:

| Propriedade | O que controla | Padrão |
|---|---|---|
| `delta.logRetentionDuration` | Por quanto tempo os arquivos do `_delta_log` (`.json`, `.checkpoint`) são mantidos | `interval 30 days` |
| `delta.deletedFileRetentionDuration` | Limite mínimo que o `VACUUM` respeita antes de remover arquivos Parquet não referenciados | `interval 7 days` |

> ⚠️ Reduzir esses valores **limita** o alcance do Time Travel.  
> Se um arquivo Parquet for removido pelo `VACUUM`, não é mais possível consultar as versões que dependiam dele.

---

## 2. Setup

Criamos a tabela e populamos com 3 inserts — cada um gerando uma versão nova no `_delta_log`.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo;
CREATE TABLE demo (
  id    INT,
  text STRING
);

In [ ]:
%%sql
INSERT INTO demo VALUES(1,'Back');
INSERT INTO demo VALUES(2,'To the');
INSERT INTO demo VALUES(3,'Future');

In [ ]:
%%sql
SELECT * FROM demo;

---
## 3. Simulando um acidente — apagando tudo

Deletamos todos os dados para simular um erro operacional que precisará ser revertido.

In [ ]:
%%sql
DELETE FROM demo;

In [ ]:
%%sql
-- Tabela vazia após o DELETE
SELECT * FROM demo;

### Consultando o histórico de versões

`DESC HISTORY` mostra todas as versões da tabela com metadados de cada operação:  
versão, timestamp, usuário, tipo de operação (`CREATE`, `WRITE`, `DELETE`...) e parâmetros.

In [ ]:
%%sql
DESCRIBE HISTORY demo;

> **Lendo a saída:** cada linha é uma versão. A versão mais recente aparece no topo.  
> Identifique a versão desejada (no nosso caso, **v3** — logo antes do `DELETE`) para usar no Time Travel.

---

## 4. Consultando versões anteriores

O Delta Lake oferece **três formas** de consultar dados de uma versão específica:

| Forma | Sintaxe | Quando usar |
|---|---|---|
| Por versão (SQL) | `demo@v3` ou `VERSION AS OF 3` | Quando você sabe o número da versão |
| Por timestamp (SQL) | `TIMESTAMP AS OF '...'` | Quando você sabe o horário aproximado |
| Por versão (Python) | `.option("versionAsOf", 3)` | Em pipelines ou DataFrames |

### 4.1 — Por número de versão (SQL)

In [ ]:
%%sql
-- Lê os dados exatamente como estavam na versão 3
-- (antes do DELETE)
SELECT * FROM demo@v3

### 4.2 — Por número de versão (Python)

In [ ]:
%python
spark.read.option('versionAsOf',3)\
.table('demo')\
.display()

### 4.3 — Por timestamp (SQL)

> Use o timestamp exato (ou aproximado) do commit desejado.  
> O valor pode ser obtido na coluna `timestamp` do `DESCRIBE HISTORY`.

In [ ]:
%%sql
-- Substitua o timestamp pelo valor encontrado no DESCRIBE HISTORY
SELECT * FROM demo TIMESTAMP AS OF '2025-05-27T17:26:23Z'

> **Importante:** consultas de Time Travel são **somente leitura** — não alteram a tabela.  
> Para de fato restaurar a tabela para um estado anterior, use `RESTORE TABLE` (próxima seção).

---

## 5. Restaurando a tabela

`RESTORE TABLE` reverte a tabela para uma versão ou timestamp específico.  
Internamente, ele grava um novo commit no `_delta_log` que aponta para os arquivos Parquet da versão desejada —  
sem deletar o histórico nem reescrever arquivos.

In [ ]:
%%sql
RESTORE TABLE demo TO VERSION AS OF 3;

In [ ]:
%%sql
SELECT * FROM demo;

In [ ]:
%%sql
-- O RESTORE gera uma nova entrada no histórico
-- O histórico anterior NÃO é apagado — o RESTORE é apenas mais uma versão
DESC HISTORY demo;

> **O que aconteceu internamente:**
>
> ```
>   versão 0   CREATE TABLE
>   versão 1   INSERT (1, 'Back')
>   versão 2   INSERT (2, 'To the')
>   versão 3   INSERT (3, 'Future')
>   versão 4   DELETE
>   versão 5   RESTORE → aponta para os parquets da v3   ◄── nova versão atual
> ```
>
> O `RESTORE` não apaga nem recria arquivos — ele apenas cria um novo commit que referencia  
> os mesmos Parquets que a versão 3 usava. O histórico completo permanece intacto.

---

## 6. VACUUM — Limpeza de arquivos obsoletos

Com o tempo, arquivos Parquet que não são mais referenciados por nenhuma versão ativa acumulam-se no storage.  
O `VACUUM` remove esses arquivos físicos, liberando espaço.

### Como o VACUUM decide o que deletar?

```
  Arquivo Parquet
       │
       ├─ Ainda referenciado em algum 'add' ativo do _delta_log?
       │       └── SIM → mantido (não importa a idade)
       │
       └─ Não referenciado (apenas em entradas 'remove')?
               ├── Idade < retention threshold → mantido (Time Travel ainda possível)
               └── Idade ≥ retention threshold → ✗ DELETADO
```

> ⚠️ **Aviso importante:** após o `VACUUM`, versões do Time Travel que dependiam dos arquivos deletados  
> **deixam de funcionar** — mesmo que o `_delta_log` ainda registre essas versões.

---

### Retenção padrão: 7 dias

Por padrão, o `VACUUM` só remove arquivos com mais de **7 dias** de inatividade.  
Isso garante uma janela segura de Time Travel sem intervenção manual.

In [ ]:
%%sql
-- Remove arquivos não referenciados com mais de 7 dias (padrão)
VACUUM demo

### `DRY RUN` — simulando antes de executar

`DRY RUN` lista os arquivos que **seriam** deletados, sem deletar nada.  
É a forma segura de verificar o impacto antes da execução final.

In [ ]:
%%sql
-- Mostra o que seria deletado — sem deletar
VACUUM demo DRY RUN

### `RETAIN 0 HOURS` — retenção zero (perigoso)

Forçar retenção zero elimina **imediatamente** todos os arquivos não referenciados, independente da idade.  
O Delta Lake tem uma proteção nativa que bloqueia retenções abaixo de 7 dias — é preciso desativá-la explicitamente.

In [ ]:
%%sql
-- ⚠️ Bloqueado por padrão: retenção < 7 dias exige desabilitar a proteção
-- DRY RUN ainda é seguro para visualizar o impacto
VACUUM demo RETAIN 0 HOURS DRY RUN

In [ ]:
%%sql
-- Desabilita a proteção de retenção mínima — use com consciência
SET spark.databricks.delta.retentionDurationCheck.enabled = False

---
### Demo — criando e limpando um arquivo desnecessário

Inserimos e deletamos uma linha para gerar um arquivo Parquet órfão,  
depois executamos o `VACUUM` para removê-lo.

In [ ]:
%%sql
INSERT INTO demo VALUES(9,'vacuum');
DELETE FROM demo WHERE id=9;

> Após o `DELETE`, o Parquet com `(9, 'vacuum')` está órfão — marcado como `remove` no `_delta_log`,  
> mas ainda presente fisicamente no disco. É exatamente esse arquivo que o `VACUUM` irá remover.

In [ ]:
%%sql
-- Confirma que o arquivo órfão aparece na lista de candidatos à remoção
VACUUM demo RETAIN 0 HOURS DRY RUN;

In [ ]:
%%sql
-- Executa a limpeza — sem volta!
VACUUM demo RETAIN 0 HOURS

---
### O que o VACUUM NÃO faz

O `VACUUM` **deleta apenas arquivos Parquet** — nunca os arquivos do `_delta_log`.

In [ ]:
%%sql
-- O _delta_log continua intacto com todas as versões registradas
-- Os .json são deletados automaticamente pelo Delta Lake após 30 dias (logRetentionDuration)
DESC HISTORY demo;

### Consequência: Time Travel de versões limpas não funciona mais

In [ ]:
%%sql
-- ❌ Erro: o _delta_log ainda registra a v8, mas o Parquet físico foi removido pelo VACUUM
-- Time Travel para versões cujos arquivos foram deletados é impossível
SELECT * FROM demo@v8

> **Por que o erro ocorre?**  
> O `_delta_log` ainda registra a versão 8 e sabe qual arquivo Parquet ela usava.  
> Mas o `VACUUM` deletou esse arquivo do disco.  
> O Delta Lake encontra a referência, tenta abrir o arquivo — e falha.
>
> ```
>   _delta_log v8  →  aponta para  →  part-xxxx.snappy.parquet
>                                              ↑
>                                        não existe mais no disco
>                                              ↓
>                                        ❌ FileNotFoundException
> ```

---

### Resumo — VACUUM vs _delta_log

```
  VACUUM                          _delta_log (limpeza automática)
  ──────────────────────────────  ──────────────────────────────────
  Remove .parquet órfãos          Remove .json e .checkpoint antigos
  Executado manualmente           Executado automaticamente
  Threshold: deletedFileRetention Threshold: logRetentionDuration
  Padrão: 7 dias                  Padrão: 30 dias
  Impacto: limita Time Travel     Impacto: limita DESC HISTORY
```

---

## 7. Configurando retenção por tabela

Os thresholds de retenção podem ser ajustados individualmente por tabela via `TBLPROPERTIES`.  
Isso permite estratégias diferentes para tabelas críticas (retenção maior) vs. tabelas temporárias (retenção menor).

| Propriedade | Controla | Impacto ao aumentar |
|---|---|---|
| `delta.deletedFileRetentionDuration` | Por quanto tempo Parquets removidos são mantidos no disco | Janela de Time Travel maior |
| `delta.logRetentionDuration` | Por quanto tempo os `.json` do `_delta_log` são mantidos | Histórico mais longo no `DESC HISTORY` |

In [ ]:
%%sql
-- Aumenta a retenção de arquivos deletados para 14 dias nesta tabela
-- (sobrescreve o padrão de 7 dias)
ALTER TABLE demo SET TBLPROPERTIES('delta.deletedFileRetentionDuration' = 'interval 14 days')

In [ ]:
%%sql
-- Confirma a propriedade em 'Table Properties'
DESC EXTENDED demo;

---
## 8. Resumo — Time Travel & Vacuum

```
┌───────────────────────────────────────────────────────────────────┐
│                   CICLO DE VIDA DE UM ARQUIVO                     │
│                                                                   │
│  INSERT/UPDATE  →  .parquet criado  →  referenciado em 'add'      │
│                                              │                    │
│  DELETE/UPDATE  →  entrada 'remove' no log   │                    │
│                                              ▼                    │
│                    .parquet agora é 'órfão' (fisicamente existe)  │
│                              │                                    │
│                    ┌─────────┴──────────┐                         │
│                    │                    │                         │
│              idade < 7 dias       idade ≥ 7 dias                  │
│            Time Travel OK         VACUUM remove                   │
│                                   Time Travel ❌                  │
└───────────────────────────────────────────────────────────────────┘
```

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Ver versões da tabela | `DESC HISTORY demo` |
| Consultar versão anterior (número) | `SELECT * FROM demo@v3` |
| Consultar versão anterior (timestamp) | `SELECT * FROM demo TIMESTAMP AS OF '...'` |
| Consultar versão anterior (Python) | `.option("versionAsOf", 3)` |
| Restaurar tabela | `RESTORE TABLE demo TO VERSION AS OF 3` |
| Simular limpeza | `VACUUM demo DRY RUN` |
| Limpar com retenção padrão | `VACUUM demo` |
| Limpar com retenção customizada | `VACUUM demo RETAIN N HOURS` |
| Configurar retenção de arquivos | `ALTER TABLE demo SET TBLPROPERTIES(...)` |